In [24]:
from kafka.admin import KafkaAdminClient, NewTopic

kafka_client: KafkaAdminClient = KafkaAdminClient(bootstrap_servers="kafka:9093")
topic = NewTopic(name="sensor-data", num_partitions=3, replication_factor=1)

# %%

In [25]:
kafka_client.list_topics()
# %%

['sensor-data']

In [26]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as f
from pyspark.sql.avro.functions import from_avro

spark = SparkSession.builder \
         .appName("KafkaStream") \
         .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.0,org.apache.spark:spark-avro_2.13:4.1.0") \
         .getOrCreate()

In [27]:
sensor_schema = """{
    "type": "record",
    "name": "SensorReading",
    "namespace": "com.example",
    "fields": [
        {"name": "timestamp", "type": "long"},
        {"name": "sensor_id", "type": "string"},
        {"name": "temperature", "type": "double"},
        {"name": "humidity", "type": "double"},
        {"name": "location", "type": "string"}
    ]
}"""

In [28]:
df_weather = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9093") \
    .option("subscribe", "sensor-data") \
    .option("startingOffsets", "earliest") \
    .load()

In [29]:
df_weather.printSchema()
df_parsed = df_weather.select(
    from_avro(f.col("value"), sensor_schema).alias("data")) \
    .select("data.*")
# %%

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [30]:
df_parsed.writeStream\
    .outputMode("append")\
    .format("console")\
    .option("truncate", "false")\
    .trigger(processingTime="5 seconds")\
    .start()

26/01/29 21:33:47 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-caf8a475-c080-46c2-b43a-b6db718c8c7f. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/01/29 21:33:47 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+----------+----------+------------------+------------------+--------+
|timestamp |sensor_id |temperature       |humidity          |location|
+----------+----------+------------------+------------------+--------+
|1769722337|sensor_100|22.242137604727436|54.628786546145825|Tokyo   |
|1769722338|sensor_10 |17.582745859890977|51.914045091067706|London  |
|1769722339|sensor_68 |15.213198359753202|57.19528167349211 |Kyiv    |
|1769722340|sensor_32 |29.678736750075224|31.337156564540475|New York|
|1769722341|sensor_83 |26.627893310936855|66.04504125603523 |New York|
|1769722342|sensor_36 |12.828920031972828|41.63047317462642 |London  |
|1769722343|sensor_78 |16.487217562248087|52.86071103871272 |London  |
|1769722344|sensor_88 |19.43871541031199 |31.81521346543635 |New York|
|1769722345|sensor_32 |11.435333316187872|44.78646427190614 |London  |
|1769722346|sensor_73 |29.042543533350237|50.303684